# Phase 5: Model Training

Training classification models

In [ ]:
"""Phase 5: Classification Model TrainingThis script trains multiple classification models on the feature-selected dataset.IMPORTANT: Uses robust error handling to prevent crashes from individual model failures."""import osimport sysimport timeimport pandas as pdimport joblibfrom sklearn.model_selection import train_test_splitfrom sklearn.linear_model import LogisticRegressionfrom sklearn.tree import DecisionTreeClassifierfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.naive_bayes import GaussianNBfrom sklearn.svm import SVCdef load_data_and_features(data_path, features_path):    """Load the feature-selected dataset and selected features list."""    print(f"\n{'='*60}")    print("STEP 1: Loading Data and Features")    print(f"{'='*60}")        # Load dataset    if not os.path.exists(data_path):        print(f"ERROR: Data file not found at {data_path}")        print("Please run pipeline/4_feature_selection.py first")        sys.exit(1)        # Load selected features    if not os.path.exists(features_path):        print(f"ERROR: Features file not found at {features_path}")        print("Please run pipeline/4_feature_selection.py first")        sys.exit(1)        try:        # Load data        df = pd.read_csv(data_path)        print(f"✓ Successfully loaded data from {data_path}")        print(f"  Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")                # Load selected features        with open(features_path, 'r') as f:            selected_features = [line.strip() for line in f if line.strip()]                print(f"\n✓ Successfully loaded {len(selected_features)} selected features from {features_path}")                # Verify FTR exists        if 'FTR' not in df.columns:            print(f"\nERROR: Target column 'FTR' not found in dataset")            print(f"Available columns: {list(df.columns)}")            sys.exit(1)                # Verify all selected features exist        missing_features = [feat for feat in selected_features if feat not in df.columns]        if missing_features:            print(f"\nERROR: The following selected features are missing from dataset:")            for feat in missing_features:                print(f"  ✗ {feat}")            sys.exit(1)                # Create X and y        X = df[selected_features].copy()        y = df['FTR'].copy()                print(f"\nFeature List ({len(selected_features)} features):")        for i, feat in enumerate(selected_features, 1):            print(f"  {i:2d}. {feat}")                print(f"\nShape of X (features): {X.shape}")        print(f"Shape of y (target):   {y.shape}")                print(f"\n✓ Step 1 completed successfully")                return X, y, selected_features            except Exception as e:        print(f"ERROR: Failed to load data: {e}")        sys.exit(1)def split_train_test(X, y):    """Split data into training and test sets with stratification."""    print(f"\n{'='*60}")    print("STEP 2: Train/Test Split")    print(f"{'='*60}")        print(f"\nSplitting data with:")    print(f"  test_size: 0.2 (20%)")    print(f"  random_state: 42 (for reproducibility)")    print(f"  stratify: y (maintain class distribution)")        try:        X_train, X_test, y_train, y_test = train_test_split(            X, y,            test_size=0.2,            random_state=42,            stratify=y        )                print(f"\n✓ Split completed successfully")        print(f"\nTraining Set Size: {len(X_train)} samples ({len(X_train)/len(X)*100:.1f}%)")        print(f"Test Set Size:     {len(X_test)} samples ({len(X_test)/len(X)*100:.1f}%)")                # Print class distribution        print(f"\nClass Distribution Comparison:")        print(f"{'='*50}")        print(f"{'Class':<15} {'Train Count':<15} {'Test Count':<15}")        print(f"{'='*50}")                train_dist = y_train.value_counts().sort_index()        test_dist = y_test.value_counts().sort_index()                class_labels = {0: 'Away Win (0)', 1: 'Draw (1)', 2: 'Home Win (2)'}                for class_val in sorted(train_dist.index):            train_count = train_dist.get(class_val, 0)            test_count = test_dist.get(class_val, 0)            train_pct = train_count / len(y_train) * 100            test_pct = test_count / len(y_test) * 100            label = class_labels.get(class_val, f'Class {class_val}')            print(f"{label:<15} {train_count:<6} ({train_pct:5.1f}%)  {test_count:<6} ({test_pct:5.1f}%)")                print(f"\n✓ Step 2 completed successfully")                return X_train, X_test, y_train, y_test            except Exception as e:        print(f"ERROR: Failed to split data: {e}")        sys.exit(1)def define_models():    """Define all classification models to train."""    print(f"\n{'='*60}")    print("STEP 3: Defining Models")    print(f"{'='*60}")        models = {        "Logistic Regression": LogisticRegression(            max_iter=1000,            random_state=42        ),        "Decision Tree": DecisionTreeClassifier(            random_state=42        ),        "Random Forest": RandomForestClassifier(            n_estimators=100,            random_state=42,            n_jobs=-1        ),        "Naive Bayes": GaussianNB(),        "SVM": SVC(            kernel="rbf",            random_state=42,            probability=True        )    }        print(f"\nModels Defined ({len(models)} models):")    print(f"{'='*50}")        for i, (name, model) in enumerate(models.items(), 1):        print(f"  {i}. {name}")        print(f"     {model}")        print(f"\n{'='*50}")    print("⚠ WARNING: SVM with RBF kernel may be slow on large datasets")    print(f"{'='*50}")    print("  - SVM training time scales poorly with dataset size")    print("  - For datasets > 10,000 samples, consider using LinearSVC")    print("  - Current implementation uses probability=True (adds overhead)")        print(f"\n✓ Step 3 completed successfully")        return modelsdef train_models(models, X_train, y_train):    """Train all models with robust error handling."""    print(f"\n{'='*60}")    print("STEP 4: Training Models")    print(f"{'='*60}")        trained_models = {}    training_times = {}    failed_models = []        print(f"\nTraining {len(models)} models on {len(X_train)} samples...")    print(f"{'='*60}")        for name, model in models.items():        print(f"\nTraining: {name}")        print(f"  Model: {model.__class__.__name__}")                try:            start_time = time.time()                        # Train the model            model.fit(X_train, y_train)                        end_time = time.time()            training_time = end_time - start_time                        # Store successfully trained model            trained_models[name] = model            training_times[name] = training_time                        print(f"  ✓ Training completed successfully")            print(f"  ⏱ Training time: {training_time:.2f} seconds")                    except Exception as e:            print(f"  ✗ Training FAILED: {e}")            failed_models.append(name)        # Summary    print(f"\n{'='*60}")    print("Training Summary:")    print(f"{'='*60}")    print(f"  Successfully trained: {len(trained_models)}/{len(models)} models")    print(f"  Failed:              {len(failed_models)}/{len(models)} models")        if trained_models:        print(f"\n✓ Successfully Trained Models:")        for i, (name, train_time) in enumerate(training_times.items(), 1):            print(f"  {i}. {name:<25} ({train_time:.2f}s)")        if failed_models:        print(f"\n✗ Failed Models:")        for i, name in enumerate(failed_models, 1):            print(f"  {i}. {name}")        if not trained_models:        print(f"\nERROR: All models failed to train")        sys.exit(1)        print(f"\n✓ Step 4 completed successfully")        return trained_models, training_timesdef save_models_and_test_data(trained_models, X_test, y_test, output_dir):    """Save all trained models and test data."""    print(f"\n{'='*60}")    print("STEP 5: Saving Models and Test Data")    print(f"{'='*60}")        # Ensure output directory exists    os.makedirs(output_dir, exist_ok=True)        saved_files = []        # Save each trained model    print(f"\nSaving Trained Models:")    print(f"{'='*50}")        for name, model in trained_models.items():        # Create filename: model_<lowercase_name>.pkl        filename = f"model_{name.lower().replace(' ', '_')}.pkl"        filepath = os.path.join(output_dir, filename)                try:            joblib.dump(model, filepath)            print(f"  ✓ {name:<25} → {filename}")            saved_files.append(filepath)        except Exception as e:            print(f"  ✗ {name:<25} → Failed: {e}")        # Save test data    print(f"\nSaving Test Data:")    print(f"{'='*50}")        # Save X_test    X_test_path = os.path.join(output_dir, 'X_test.csv')    try:        X_test.to_csv(X_test_path, index=False)        print(f"  ✓ X_test saved → X_test.csv")        print(f"    Shape: {X_test.shape}")        saved_files.append(X_test_path)    except Exception as e:        print(f"  ✗ X_test save failed: {e}")        # Save y_test    y_test_path = os.path.join(output_dir, 'y_test.csv')    try:        y_test_df = pd.DataFrame({'FTR': y_test})        y_test_df.to_csv(y_test_path, index=False)        print(f"  ✓ y_test saved → y_test.csv")        print(f"    Shape: {y_test_df.shape}")        saved_files.append(y_test_path)    except Exception as e:        print(f"  ✗ y_test save failed: {e}")        print(f"\n✓ Step 5 completed successfully")    print(f"\nTotal Files Saved: {len(saved_files)}")        return saved_filesprint("\n" + "="*60)print("EPL MATCH DATA - MODEL TRAINING")print("="*60)    # Define pathsdata_path = "pipeline/output/feature_selected_data.csv"features_path = "pipeline/output/selected_features.txt"output_dir = "pipeline/output"    # Step 1: Load data and featuresX, y, selected_features = load_data_and_features(data_path, features_path)    # Step 2: Train/test splitX_train, X_test, y_train, y_test = split_train_test(X, y)    # Step 3: Define modelsmodels = define_models()    # Step 4: Train modelstrained_models, training_times = train_models(models, X_train, y_train)    # Step 5: Save models and test datasaved_files = save_models_and_test_data(trained_models, X_test, y_test, output_dir)    print(f"\n{'='*60}")print("✓ ALL MODEL TRAINING STEPS COMPLETED!")print(f"{'='*60}")print(f"\nSummary:")print(f"  Models trained:  {len(trained_models)}")print(f"  Files saved:     {len(saved_files)}")print(f"\nModel Files (.pkl):")for name in trained_models.keys():    filename = f"model_{name.lower().replace(' ', '_')}.pkl"    print(f"  - {filename}")print(f"\nTest Data Files (.csv):")print(f"  - X_test.csv")print(f"  - y_test.csv")print(f"\nYou can now proceed to Phase 6: Model Evaluation")print("="*60 + "\n")